# MVP simplificado · Riesgo de no renovación de membresías (Mtabolic)

Este notebook es una **versión corta y comentada** de `MVP.ipynb`. Hace **exactamente lo mismo que la aplicación** (mismo corte T−30, mismas 8 variables, mismo Random Forest, mismo umbral, mismos niveles y mismos motivos SHAP), pero sin la historia del proyecto: sin las comparaciones antes/después de M01, sin el modelo de 10 variables y con pocos gráficos.

**Cómo usarlo**
1. Los libros `Control Gym 2025*.xlsx` y `Control Gym 2026*.xlsx` deben estar en la carpeta `plantillas/` del proyecto (o en la carpeta indicada en la variable de entorno `MTABOLIC_DATOS`).
2. Menú **Run → Run All Cells** (Ejecutar todo). Tarda alrededor de un minuto.
3. Al final, la **Parte 11 · Controles** comprueba que las cifras coinciden con el notebook original y con la app. Si todo dice «OK», el notebook está listo para trabajar sobre él.

**Mapa: qué parte del notebook equivale a qué parte de la app**

| Parte | Qué hace | Dónde está en la app (`Backend/app/...`) |
|---|---|---|
| 2 | Leer las hojas mensuales | `infrastructure/historico/lector_libros.py` |
| 3 | Limpiar identidad, plan y fechas | `domain/historico/normalizacion.py` |
| 4 | Estado del ciclo y etiqueta T+30 | `domain/historico/ciclos.py` |
| 5 | Variables en T−30 | `domain/modelo/variables.py` |
| 6 | Partición con márgenes y medianas de entrenamiento | `domain/modelo/reentrenamiento.py` |
| 7 | Pipeline y Random Forest | `infrastructure/ml/random_forest.py` |
| 10 | Niveles 0,70 / 0,50 y motivos SHAP | `domain/modelo/umbrales.py` y `random_forest.py` |

Ningún resultado muestra nombres ni cédulas: los socios se identifican con un **código anónimo** (`MTB-XXXXXXXX`).

## Parte 1 · Configuración
Librerías y **parámetros del negocio**. Todos los números fijos del proyecto están aquí, en un solo lugar.

In [ ]:
# --- Librerías estándar ---
import os, re, hmac, hashlib, secrets, unicodedata, warnings
from pathlib import Path
from datetime import datetime

# --- Análisis de datos ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# --- Aprendizaje automático ---
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             average_precision_score, roc_auc_score, brier_score_loss)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import shap                                   # explicabilidad (valores SHAP)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", None)

# Las versiones deben ser las mismas que usa la app; con otras versiones las cifras pueden cambiar
print("scikit-learn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__, "| shap", shap.__version__)
assert sklearn.__version__ == "1.8.0" and shap.__version__ == "0.54.1", "Instalar scikit-learn==1.8.0 y shap==0.54.1"

In [ ]:
# ===================== PARÁMETROS DEL NEGOCIO (iguales a la app) =====================
SEMILLA = 42                      # número fijo: todo lo aleatorio se repite igual en cada ejecución
DIAS_ANTES_PREDICCION = 30        # T−30: la lista se genera 30 días antes del vencimiento
DIAS_OBSERVACION = 30             # T+30: se espera 30 días después del vencimiento para saber si renovó
TOP_K = 25                        # capacidad de contacto: socios que se llaman primero cada mes
FECHA_MINIMA_RAZONABLE = pd.Timestamp("2024-12-01")   # vencimientos anteriores se consideran error de digitación
DURACION_MIN, DURACION_MAX = 21, 400                  # duraciones fuera de este rango son imposibles

# Partición por fecha de vencimiento, con márgenes de 30 días que se descartan (mejora M01)
INICIO_VALIDACION = pd.Timestamp("2026-01-01")
INICIO_PRUEBA = pd.Timestamp("2026-05-01")
DIAS_MARGEN = 30

# Las 8 variables del modelo (sin forma de pago ni fecha de inicio aproximada, decisión M01)
VARIABLES_NUMERICAS = ["ciclos_previos", "antiguedad_dias", "duracion_modelo",
                       "monto_actual", "monto_historico", "dias_desde_ultima_membresia"]
VARIABLES_CATEGORICAS = ["plan", "mes_vencimiento"]
VARIABLES_MODELO = VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS

# Random Forest: mismos hiperparámetros que la app (fijados antes de ver resultados, sin búsqueda)
HIPERPARAMETROS_RF = dict(n_estimators=300, max_depth=6, min_samples_leaf=20,
                          class_weight="balanced", random_state=SEMILLA)

# Niveles de riesgo de la app: ALTO si score >= 0,70; MEDIO si >= 0,50; si no, BAJO
UMBRAL_ALTO, UMBRAL_MEDIO = 0.70, 0.50
DURACION_ESTANDAR = {"MENSUAL": 30, "TRIMESTRAL": 90, "SEMESTRAL": 180, "ANUAL": 365, "OTRO": 30}

In [ ]:
# Carpeta de los libros de Excel: variable de entorno MTABOLIC_DATOS o, por defecto, ../plantillas
CARPETA_DATOS = Path(os.environ.get("MTABOLIC_DATOS", Path("..") / "plantillas"))

def buscar_archivo(anio):
    candidatos = sorted(CARPETA_DATOS.glob(f"*Control*Gym*{anio}*.xlsx"))
    if not candidatos:
        raise FileNotFoundError(f"No se encontró el libro {anio} en {CARPETA_DATOS}")
    return candidatos[0]

archivo_2025, archivo_2026 = buscar_archivo(2025), buscar_archivo(2026)
print("Libros encontrados:", archivo_2025.name, "|", archivo_2026.name)

# Código anónimo del socio: HMAC-SHA256 con una sal secreta (la misma del notebook original).
# Sin la sal no se puede volver de un código a una cédula. La sal nunca se imprime.
ARCHIVO_SAL = Path("..") / "salidas" / ".sal_codigos"
SAL = os.environ.get("MTABOLIC_SAL_CODIGOS", "").strip()
if not SAL and ARCHIVO_SAL.is_file():
    SAL = ARCHIVO_SAL.read_text(encoding="utf-8").strip()
if not SAL:
    SAL = secrets.token_hex(32)
    ARCHIVO_SAL.parent.mkdir(parents=True, exist_ok=True)
    ARCHIVO_SAL.write_text(SAL, encoding="utf-8")

def codigo_socio(socio_id):
    return "MTB-" + hmac.new(SAL.encode(), str(socio_id).encode(), hashlib.sha256).hexdigest()[:8].upper()

## Parte 2 · Leer los libros de Excel
Cada libro tiene muchas hojas (diarias, novedades, horas extras…). Solo se usan las **20 hojas mensuales** (ENERO…AGOSTO). En cada hoja se busca la fila de títulos y se identifica cada columna aunque cambie de nombre o de posición.

In [ ]:
MESES = {"ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6, "JULIO": 7,
         "AGOSTO": 8, "SEPTIEMBRE": 9, "OCTUBRE": 10, "NOVIEMBRE": 11, "DICIEMBRE": 12}

# Nombres posibles de cada columna en los Excel (ya en mayúsculas y sin tildes)
ALIAS_COLUMNAS = {
    "nombre":        ["NOMBRES Y APELLIDOS", "NOMBRES Y APELLIDOS DEL SOCIO"],
    "cedula":        ["CEDULA"],
    "plan":          ["TIPO DE PLAN"],
    "fecha_inicio":  ["FECHA INSCRIPCION", "FECHA INICIO"],
    "fecha_fin":     ["FECHA FIN PLAN", "FECHA FIN"],
    "monto_cobrado": ["MONTO A COBRAR", "MONTO COBRAR"],
}

def texto_normalizado(valor):
    # " Cédula  " -> "CEDULA": mayúsculas, sin tildes y con espacios simples
    if pd.isna(valor):
        return ""
    texto = unicodedata.normalize("NFKD", str(valor).strip().upper()).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", texto)

def leer_hoja_mensual(ruta, hoja, anio):
    tabla = pd.read_excel(ruta, sheet_name=hoja, header=None)
    # 1) La fila de títulos es la primera (de las 10 primeras) que contiene "CEDULA" y "NOMBRE"
    fila_titulos = next(i for i in range(min(10, len(tabla)))
                        if "CEDULA" in " | ".join(map(texto_normalizado, tabla.iloc[i]))
                        and "NOMBRE" in " | ".join(map(texto_normalizado, tabla.iloc[i])))
    titulos = [texto_normalizado(v) for v in tabla.iloc[fila_titulos]]
    datos = tabla.iloc[fila_titulos + 1:]
    salida = pd.DataFrame(index=datos.index)
    # 2) Para cada campo, la primera columna cuyo título contiene alguno de sus alias
    for campo, alias in ALIAS_COLUMNAS.items():
        indice = next((i for i, t in enumerate(titulos) if any(a in t for a in alias)), None)
        salida[campo] = datos.iloc[:, indice] if indice is not None else np.nan
    salida["mes_hoja"] = pd.Timestamp(anio, MESES[hoja.strip().upper()], 1)   # primer día del mes de la hoja
    return salida

tablas = []
for ruta, anio in [(archivo_2025, 2025), (archivo_2026, 2026)]:
    for hoja in pd.ExcelFile(ruta).sheet_names:
        if hoja.strip().upper() in MESES:            # solo hojas mensuales
            tablas.append(leer_hoja_mensual(ruta, hoja, anio))
crudo = pd.concat(tablas, ignore_index=True)
print("Hojas mensuales leídas:", len(tablas), "| filas leídas (con vacías y totales):", len(crudo))

## Parte 3 · Limpiar: filas útiles, identidad del socio, fechas y plan
- Se descartan filas vacías, de totales y títulos repetidos.
- **Identidad del socio:** la cédula (solo dígitos, 9 o más); si no hay, el nombre normalizado. Así se unen todas las membresías de una misma persona.
- Las fechas y montos escritos de distintas formas se convierten a un formato único.
- El plan, escrito de **281 formas**, se reduce a 6 categorías.

In [ ]:
# --- 3.1 Filas útiles ---
nombre = crudo["nombre"].astype("string").fillna("").str.strip()
es_util = (nombre.ne("")
           & ~nombre.str.upper().str.contains("TOTAL|SUBTOTAL|SUMA|MONTO", regex=True)
           & ~nombre.str.upper().isin(["STATUS", "NOMBRES Y APELLIDOS", "NOMBRES Y APELLIDOS DEL SOCIO"]))
datos = crudo.loc[es_util].copy()
print("Filas útiles (registros de socios):", len(datos))

# --- 3.2 Conversión de fechas y montos ---
def convertir_fecha(valor):
    if pd.isna(valor):
        return pd.NaT
    if isinstance(valor, (pd.Timestamp, datetime)):
        return pd.Timestamp(valor).normalize()
    if isinstance(valor, (int, float)) and not isinstance(valor, bool):
        if 20000 <= float(valor) <= 60000:                  # número interno de Excel
            return (pd.Timestamp("1899-12-30") + pd.to_timedelta(float(valor), unit="D")).normalize()
    texto = str(valor).strip()
    if not texto:
        return pd.NaT
    fecha = pd.to_datetime(texto, errors="coerce", dayfirst=True)
    if pd.isna(fecha):
        fecha = pd.to_datetime(texto, errors="coerce", dayfirst=False)
    return fecha.normalize() if not pd.isna(fecha) else pd.NaT

def convertir_numero(valor):
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)) and not isinstance(valor, bool):
        return float(valor)
    hallado = re.search(r"-?\d+(?:\.\d+)?", str(valor).replace(",", "."))   # "39,50 USD" -> 39.5
    return float(hallado.group(0)) if hallado else np.nan

datos["fecha_inicio"] = datos["fecha_inicio"].apply(convertir_fecha)
datos["fecha_fin"] = datos["fecha_fin"].apply(convertir_fecha)            # fecha_fin = T (vencimiento)
datos["monto_cobrado"] = datos["monto_cobrado"].apply(convertir_numero)
datos["plan_crudo"] = datos["plan"].fillna("").astype(str).str.strip().str.upper()

# --- 3.3 Identidad del socio ---
def normalizar_nombre(texto):
    texto = unicodedata.normalize("NFKD", "" if pd.isna(texto) else str(texto)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z ]", " ", texto.upper())).strip()

def normalizar_cedula(valor):
    digitos = "" if pd.isna(valor) else re.sub(r"\D", "", str(valor))
    return digitos if len(digitos) >= 9 else np.nan

datos["nombre_normalizado"] = datos["nombre"].apply(normalizar_nombre)
datos = datos.loc[datos["nombre_normalizado"].str.len() >= 5].copy()      # identidad utilizable
cedula = datos["cedula"].apply(normalizar_cedula)
datos["socio_id"] = np.where(cedula.notna(), "C:" + cedula.fillna(""), "N:" + datos["nombre_normalizado"])
print("Registros con identidad:", len(datos), "| socios distintos:", datos["socio_id"].nunique())

# --- 3.4 Duración y plan normalizado ---
datos["duracion_dias"] = (datos["fecha_fin"] - datos["fecha_inicio"]).dt.days
FORMAS_PAGO_EN_CAMPO_PLAN = ("DATAFAST", "TRANSF", "EFECTIVO", "TARJETA", "DEPOSITO")

def normalizar_plan(texto, duracion):
    texto = str(texto or "").upper().strip()
    if any(p in texto for p in FORMAS_PAGO_EN_CAMPO_PLAN):   # el campo traía una forma de pago, no un plan
        texto = ""
    if "DIARIO" in texto: return "DIARIO"
    if "ANUAL" in texto or "AÑO" in texto or "ANO" in texto: return "ANUAL"
    if "SEMESTR" in texto or "6 MES" in texto: return "SEMESTRAL"
    if any(x in texto for x in ("3 MES", "TRES MES", "TRIMEST")): return "TRIMESTRAL"
    if "MENSUAL" in texto or re.search(r"\b1\s*MES\b", texto) or "PLAN MES" in texto: return "MENSUAL"
    if pd.notna(duracion):                                   # sin palabra clave: se usa la duración
        if 21 <= duracion <= 45: return "MENSUAL"
        if 80 <= duracion <= 130: return "TRIMESTRAL"
        if 160 <= duracion <= 200: return "SEMESTRAL"
        if 330 <= duracion <= 400: return "ANUAL"
    return "OTRO"

datos["plan"] = [normalizar_plan(t, d) for t, d in zip(datos["plan_crudo"], datos["duracion_dias"])]
print("Escrituras distintas del plan:", datos["plan_crudo"].nunique(), "-> categorías:", sorted(datos["plan"].unique()))

## Parte 4 · Estado de cada membresía y etiqueta (¿renovó entre T y T+30?)
- **Fecha de corte:** último día del último mes con datos (31-08-2026).
- Cada membresía queda **evaluable**, **pendiente** (aún no pasan 30 días desde su vencimiento) o **excluida** (con su motivo).
- **Etiqueta `no_renovo`:** 1 si el socio no registró otra membresía (que no sea pase diario) con fecha de actividad entre T y T+30; 0 si sí.
- **Fecha de actividad:** la fecha de inicio o, si falta, el primer día del mes de la hoja.

In [ ]:
fecha_corte = datos["mes_hoja"].max() + pd.offsets.MonthEnd(0)
datos["fecha_limite_observacion"] = datos["fecha_fin"] + pd.Timedelta(days=DIAS_OBSERVACION)   # T+30
datos["fecha_prediccion"] = datos["fecha_fin"] - pd.Timedelta(days=DIAS_ANTES_PREDICCION)       # T−30
datos["fecha_actividad"] = datos["fecha_inicio"].fillna(datos["mes_hoja"])
datos["mes_vencimiento"] = datos["fecha_fin"].dt.to_period("M").astype("string")
print("Fecha de corte de los datos:", fecha_corte.date())

def estado_ciclo(f):
    # Las reglas se aplican en este orden; gana la primera que se cumple (igual que la app)
    if pd.isna(f["fecha_fin"]):                     return "EXCLUIDO - Sin fecha de vencimiento"
    if f["fecha_fin"] < FECHA_MINIMA_RAZONABLE:     return "EXCLUIDO - Fecha fuera de rango"
    if f["fecha_fin"] > fecha_corte:                return "EXCLUIDO - Vencimiento posterior al período disponible"
    if f["plan"] == "DIARIO":                       return "EXCLUIDO - Pase diario"
    if pd.notna(f["duracion_dias"]) and (f["duracion_dias"] <= DURACION_MIN or f["duracion_dias"] > DURACION_MAX):
        return "EXCLUIDO - Duración implausible"
    if f["fecha_limite_observacion"] > fecha_corte: return "PENDIENTE - Ventana T+30 incompleta"
    return "EVALUABLE"

datos["estado_ciclo"] = datos.apply(estado_ciclo, axis=1)

# Etiqueta: buscar otra membresía del mismo socio entre T y T+30
membresias = datos.loc[datos["plan"].ne("DIARIO") & datos["fecha_actividad"].notna()]
actividades_por_socio = {s: g.sort_values("fecha_actividad") for s, g in membresias.groupby("socio_id")}

def renovo(fila):
    h = actividades_por_socio.get(fila["socio_id"])
    if h is None:
        return False
    en_ventana = ((h.index != fila.name) & (h["fecha_actividad"] >= fila["fecha_fin"])
                  & (h["fecha_actividad"] <= fila["fecha_limite_observacion"]))
    return bool(en_ventana.any())

evaluable = datos["estado_ciclo"].eq("EVALUABLE")
datos["no_renovo"] = np.nan
datos.loc[evaluable, "no_renovo"] = [0 if renovo(f) else 1 for _, f in datos.loc[evaluable].iterrows()]

display(datos["estado_ciclo"].value_counts().rename_axis("estado").reset_index(name="registros"))
n_eval, n_nr = int(evaluable.sum()), int((datos["no_renovo"] == 1).sum())
print(f"Membresías evaluables: {n_eval} | no renovaron: {n_nr} ({100 * n_nr / n_eval:.1f} %)")

## Parte 5 · Las variables que ve el modelo (solo información hasta T−30)
Para cada membresía, su **historial admitido** son las otras membresías no excluidas del mismo socio con fecha de actividad **anterior** a la del ciclo **y menor o igual a T−30**. Lo que pasa después de T−30 no existe para el modelo: así se evita la **fuga de información**.

In [ ]:
# Membresías con calidad suficiente (evaluables + pendientes), ordenadas por socio y fecha.
# El orden importa: el Random Forest sortea filas y con otro orden los resultados cambian ligeramente.
modelables = (datos.loc[~datos["estado_ciclo"].str.startswith("EXCLUIDO")]
              .sort_values(["socio_id", "fecha_actividad", "fecha_fin"]).copy())

def variables_historicas(fila, historial, corte_t30=True):
    # Historial admitido: otras membresías del socio, anteriores al ciclo y (con corte) con actividad <= T−30
    previos = historial.loc[(historial.index != fila.name) & (historial["fecha_actividad"] < fila["fecha_actividad"])]
    if corte_t30:
        previos = previos.loc[previos["fecha_actividad"] <= fila["fecha_prediccion"]]
    if previos.empty:                           # socio nuevo: sin historial
        dias_desde, primera = np.nan, fila["fecha_actividad"]
    else:
        dias_desde = (fila["fecha_actividad"] - previos["fecha_actividad"].max()).days
        primera = min(previos["fecha_actividad"].min(), fila["fecha_actividad"])
    return {"ciclos_previos": len(previos),
            "monto_historico": previos["monto_cobrado"].fillna(0).sum(),
            "dias_desde_ultima_membresia": dias_desde,
            "antiguedad_dias": max(0, (fila["fecha_prediccion"] - primera).days)}

def calcular_historial(ciclos, base, corte_t30=True):
    por_socio = {s: g for s, g in base.groupby("socio_id")}
    vacio = base.iloc[0:0]
    return pd.DataFrame([variables_historicas(f, por_socio.get(f["socio_id"], vacio), corte_t30)
                         for _, f in ciclos.iterrows()], index=ciclos.index)

modelables = modelables.join(calcular_historial(modelables, modelables))
print("Membresías con variables calculadas:", len(modelables))

In [ ]:
# Comprobación anti-fuga (versión simple de la prueba §7.5 del original):
# se BORRA todo lo que cada socio registró después del T−30 de cada membresía y se recalculan sus variables.
# Con el corte T−30 ninguna variable puede cambiar (debe dar 0). Sin el corte (regla anterior del MVP) cambia 1 de las
# 1.660: la fuga era posible por diseño, pero casi no ocurría en estos datos (igual que reporta M01 §9).
HISTORIAS = ["ciclos_previos", "monto_historico", "dias_desde_ultima_membresia", "antiguedad_dias"]
def cambian(corte_t30):
    referencia = calcular_historial(modelables, modelables, corte_t30)
    n = 0
    for i, f in modelables.iterrows():
        h = modelables.loc[modelables["socio_id"] == f["socio_id"]]
        h = h.loc[(h["fecha_actividad"] <= f["fecha_prediccion"]) | (h.index == i)]   # sin el "futuro"
        nuevo = pd.Series(variables_historicas(f, h, corte_t30))[HISTORIAS]
        viejo = referencia.loc[i, HISTORIAS]
        n += int(not all((pd.isna(a) and pd.isna(b)) or a == b for a, b in zip(nuevo, viejo)))
    return n
fuga_con_corte, fuga_sin_corte = cambian(True), cambian(False)
print(f"Membresías cuyas variables cambian al borrar el futuro: con corte T−30 = {fuga_con_corte} | "
      f"sin corte (regla anterior) = {fuga_sin_corte}")
assert fuga_con_corte == 0, "Hay fuga: alguna variable depende de datos posteriores a T−30"

## Parte 6 · Separar el pasado del futuro: entrenamiento, validación y prueba
Se divide por **fecha de vencimiento**, nunca al azar, con **márgenes de 30 días** descartados entre conjuntos. El monto faltante se completa con la **mediana del plan calculada solo con entrenamiento**.

In [ ]:
def asignar_particion(t):
    if t < INICIO_VALIDACION - pd.Timedelta(days=DIAS_MARGEN): return "1_entrenamiento"   # vence hasta el 01-12-2025
    if t < INICIO_VALIDACION:                                   return "M1_margen"
    if t < INICIO_PRUEBA - pd.Timedelta(days=DIAS_MARGEN):      return "2_validacion"      # 01-01 a 31-03-2026
    if t < INICIO_PRUEBA:                                       return "M2_margen"
    return "3_prueba"                                                                      # desde el 01-05-2026

modelables["particion"] = np.where(modelables["no_renovo"].notna(),
                                   modelables["fecha_fin"].apply(asignar_particion), "sin_etiqueta")

def calcular_medianas(ciclos):
    con_monto = ciclos.loc[ciclos["monto_cobrado"].notna()]
    return con_monto.groupby("plan")["monto_cobrado"].median(), con_monto["monto_cobrado"].median()

def completar_monto_y_duracion(ciclos, mediana_por_plan, mediana_general):
    # monto faltante -> mediana del plan (y si no, la general); duración faltante -> duración estándar del plan
    ciclos = ciclos.copy()
    ciclos["monto_actual"] = ciclos["monto_cobrado"].fillna(ciclos["plan"].map(mediana_por_plan)).fillna(mediana_general)
    ciclos["duracion_modelo"] = ciclos["duracion_dias"].fillna(ciclos["plan"].map(DURACION_ESTANDAR))
    for c in VARIABLES_CATEGORICAS:
        ciclos[c] = ciclos[c].astype(str).astype(object)
    return ciclos

# Medianas SOLO con entrenamiento (NC1): se aplican tal cual a validación y prueba
mediana_por_plan, mediana_general = calcular_medianas(modelables.loc[modelables["particion"] == "1_entrenamiento"])
modelables = completar_monto_y_duracion(modelables, mediana_por_plan, mediana_general)
print("Medianas de monto por plan (solo entrenamiento, USD):", mediana_por_plan.round(2).to_dict())

analitico = modelables.loc[modelables["no_renovo"].notna()].copy()     # membresías con resultado conocido
analitico["no_renovo"] = analitico["no_renovo"].astype(int)
train = analitico.loc[analitico["particion"] == "1_entrenamiento"]
validation = analitico.loc[analitico["particion"] == "2_validacion"]
test = analitico.loc[analitico["particion"] == "3_prueba"]
X_train, y_train = train[VARIABLES_MODELO], train["no_renovo"]
X_validation, y_validation = validation[VARIABLES_MODELO], validation["no_renovo"]
X_test, y_test = test[VARIABLES_MODELO], test["no_renovo"]

display(analitico.groupby("particion").agg(membresias=("no_renovo", "size"), socios=("socio_id", "nunique"),
                                           no_renovo=("no_renovo", "mean"), desde=("fecha_fin", "min"),
                                           hasta=("fecha_fin", "max")).round(3))
# Controles: los márgenes se respetan y ninguna membresía está en dos conjuntos
assert train["fecha_fin"].max() + pd.Timedelta(days=DIAS_MARGEN) < validation["fecha_fin"].min()
assert validation["fecha_fin"].max() + pd.Timedelta(days=DIAS_MARGEN) < test["fecha_fin"].min()

## Parte 7 · Modelos: referencia sin modelo, Regresión Logística y Random Forest
- **Pipeline:** la preparación (imputar, escalar, one-hot) y el modelo forman un solo objeto. Al entrenar con `fit(X_train)`, **todo** aprende solo de entrenamiento.
- **Umbral:** el que da el mejor F1 en validación.
- **Regla de selección (fijada antes de ver resultados):** el modelo debe superar a la referencia en F1, PR-AUC y Brier a la vez en validación.

In [ ]:
def crear_pipeline(modelo):
    preparacion = ColumnTransformer([
        ("numericas", Pipeline([("imputar", SimpleImputer(strategy="median")),        # vacío -> mediana de TRAIN
                                ("escalar", StandardScaler())]), VARIABLES_NUMERICAS),
        ("categoricas", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),   # vacío -> moda de TRAIN
                                  ("one_hot", OneHotEncoder(handle_unknown="ignore"))]), VARIABLES_CATEGORICAS),
    ])
    return Pipeline([("preparacion", preparacion), ("modelo", modelo)])

modelos = {
    "Regresión Logística": crear_pipeline(LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEMILLA)),
    "Random Forest": crear_pipeline(RandomForestClassifier(**HIPERPARAMETROS_RF)),
}
for m in modelos.values():
    m.fit(X_train, y_train)                    # aquí ocurre el aprendizaje, solo con entrenamiento

def metricas(y, p, umbral):
    y, p = np.asarray(y), np.asarray(p)
    pred = (p >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    top = y[np.argsort(p)[::-1][:TOP_K]].mean()
    return {"Umbral": umbral, "PR-AUC": average_precision_score(y, p), "ROC-AUC": roc_auc_score(y, p),
            "Brier": brier_score_loss(y, p), "F1": f1_score(y, pred, zero_division=0),
            "Precisión": precision_score(y, pred, zero_division=0), "Recall": recall_score(y, pred, zero_division=0),
            "Especificidad": tn / (tn + fp) if tn + fp else np.nan, "Accuracy": accuracy_score(y, pred),
            f"Precisión@{TOP_K}": top, f"Lift@{TOP_K}": top / y.mean()}

def umbral_f1(modelo):
    p = modelo.predict_proba(X_validation)[:, 1]
    tabla = pd.DataFrame([{"umbral": round(float(u), 2), "f1": f1_score(y_validation, (p >= u).astype(int), zero_division=0),
                           "precision": precision_score(y_validation, (p >= u).astype(int), zero_division=0),
                           "recall": recall_score(y_validation, (p >= u).astype(int), zero_division=0)}
                          for u in np.arange(0.20, 0.81, 0.01)])
    return float(tabla.sort_values(["f1", "precision", "recall"], ascending=False).iloc[0]["umbral"])

umbrales = {n: umbral_f1(m) for n, m in modelos.items()}
prevalencia_train = y_train.mean()             # la referencia da a todos esta probabilidad

def comparar(X, y):
    piso = metricas(y, np.full(len(y), prevalencia_train), 0.50)
    piso.update({f"Precisión@{TOP_K}": y.mean(), f"Lift@{TOP_K}": 1.0})
    filas = {"Sin modelo (piso)": piso}
    filas.update({n: metricas(y, m.predict_proba(X)[:, 1], umbrales[n]) for n, m in modelos.items()})
    return pd.DataFrame(filas).T

comparacion_validacion = comparar(X_validation, y_validation)
piso = comparacion_validacion.iloc[0]
cumple = {n: bool(f["F1"] > piso["F1"] and f["PR-AUC"] > piso["PR-AUC"] and f["Brier"] < piso["Brier"])
          for n, f in comparacion_validacion.iloc[1:].iterrows()}
candidatos = [n for n, ok in cumple.items() if ok] or list(modelos)
modelo_seleccionado_nombre = max(candidatos, key=lambda n: comparacion_validacion.loc[n, "PR-AUC"])
modelo_seleccionado = modelos[modelo_seleccionado_nombre]
umbral_seleccionado = umbrales[modelo_seleccionado_nombre]

print("VALIDACIÓN (se usa para elegir)")
display(comparacion_validacion.round(3))
print("¿Cumple los 3 criterios?", cumple, "-> seleccionado:", modelo_seleccionado_nombre, "| umbral:", umbral_seleccionado)

In [ ]:
# PRUEBA: se mira una sola vez, después de elegir. No se usa para cambiar ninguna decisión.
comparacion_prueba = comparar(X_test, y_test)
print("PRUEBA (277 membresías de mayo a agosto de 2026)")
display(comparacion_prueba.round(3))

proba_test = {n: m.predict_proba(X_test)[:, 1] for n, m in modelos.items()}
p_rf = proba_test[modelo_seleccionado_nombre]
matriz = pd.DataFrame(confusion_matrix(y_test, (p_rf >= umbral_seleccionado).astype(int), labels=[0, 1]),
                      index=["Real: renovó", "Real: no renovó"], columns=["Dice: renueva", "Dice: no renueva"])
print(f"Matriz de confusión del {modelo_seleccionado_nombre} con umbral {umbral_seleccionado}:")
display(matriz)

## Parte 8 · ¿La mejora es real? Bootstrap por socio
Un socio puede tener varias membresías en prueba (277 membresías de 215 socios) y no son independientes. Por eso en cada una de las 1.000 repeticiones se sortean **socios completos** con todas sus membresías. Se calcula la diferencia de **PR-AUC** y de **Precisión@25** (lo que pide NC2). Si el intervalo del 95 % no incluye el 0, la mejora no es casualidad.

**Cómo leer la tabla:** si el intervalo del 95 % (columnas inferior y superior) queda entero por encima de 0, el primer modelo es mejor; entero por debajo, es peor; si incluye el 0, no se puede afirmar diferencia. La Precisión@25 depende de solo 25 casos, por eso su intervalo es mucho más ancho que el de la PR-AUC.

In [ ]:
def precision_k(y, p, k=TOP_K):
    return y[np.argsort(-p, kind="mergesort")[:k]].mean()

def bootstrap_por_socio(conjunto, y, probas, n=1000):
    rng = np.random.default_rng(SEMILLA)
    socios, y = conjunto["socio_id"].to_numpy(), np.asarray(y)
    unicos = pd.unique(socios)
    filas_de = [np.flatnonzero(socios == s) for s in unicos]
    rf, rl = probas["Random Forest"], probas["Regresión Logística"]
    reps = []
    for _ in range(n):
        idx = np.concatenate([filas_de[j] for j in rng.integers(0, len(unicos), len(unicos))])
        yb = y[idx]
        if yb.min() == yb.max():               # réplica con una sola clase: no se puede medir
            continue
        reps.append({"PR-AUC: RF − sin modelo": average_precision_score(yb, rf[idx]) - yb.mean(),
                     "PR-AUC: RL − sin modelo": average_precision_score(yb, rl[idx]) - yb.mean(),
                     "PR-AUC: RF − RL": average_precision_score(yb, rf[idx]) - average_precision_score(yb, rl[idx]),
                     "Precisión@25: RF − sin modelo": precision_k(yb, rf[idx]) - yb.mean(),
                     "Precisión@25: RF − RL": precision_k(yb, rf[idx]) - precision_k(yb, rl[idx])})
    reps = pd.DataFrame(reps)
    return pd.DataFrame({"media": reps.mean(), "IC 95 % inferior": reps.quantile(0.025),
                         "IC 95 % superior": reps.quantile(0.975), "% repeticiones > 0": 100 * (reps > 0).mean(),
                         "réplicas válidas": len(reps)})

bootstrap_prueba = bootstrap_por_socio(test, y_test, proba_test)
display(bootstrap_prueba.round(3))

## Parte 9 · ¿En qué se fija el modelo? Permutación y SHAP
- **Permutación:** cuánto empeora el orden (PR-AUC) en validación si se desordena cada variable.
- **SHAP (TreeExplainer):** cuánto sube o baja cada variable la probabilidad de cada socio. Probabilidad = valor base + suma de aportes.

In [ ]:
r = permutation_importance(modelo_seleccionado, X_validation, y_validation, scoring="average_precision",
                           n_repeats=10, random_state=SEMILLA)
importancia_permutacion = (pd.DataFrame({"variable": VARIABLES_MODELO, "importancia_media": r.importances_mean,
                                         "desviacion": r.importances_std})
                           .sort_values("importancia_media", ascending=False).reset_index(drop=True))

def explicar_shap(modelo, X):
    # SHAP exacto del bosque sobre la matriz preparada; las columnas one-hot se suman a su variable original
    prep, bosque = modelo.named_steps["preparacion"], modelo.named_steps["modelo"]
    matriz = prep.transform(X)
    matriz = np.asarray(matriz.toarray() if hasattr(matriz, "toarray") else matriz, dtype=np.float64)
    explicador = shap.TreeExplainer(bosque)
    valores = explicador.shap_values(matriz)
    valores = np.stack(valores, axis=-1) if isinstance(valores, list) else np.asarray(valores)
    por_columna = valores[:, :, 1]                               # clase 1 = "no renovó"
    base = float(np.atleast_1d(explicador.expected_value)[1])
    destino = []                                                 # variable original de cada columna
    for nombre, transformador, columnas in prep.transformers_:
        if nombre == "numericas":
            destino += list(columnas)
        elif nombre == "categoricas":
            for col, cats in zip(columnas, transformador.named_steps["one_hot"].categories_):
                destino += [col] * len(cats)
    agregado = np.zeros((matriz.shape[0], len(VARIABLES_MODELO)))
    for j, v in enumerate(destino):                              # suma en el mismo orden que la app
        agregado[:, VARIABLES_MODELO.index(v)] += por_columna[:, j]
    error = float(np.max(np.abs(base + agregado.sum(axis=1) - modelo.predict_proba(X)[:, 1])))
    return base, pd.DataFrame(agregado, index=X.index, columns=VARIABLES_MODELO), error

base_shap, shap_validacion, error_shap = explicar_shap(modelo_seleccionado, X_validation)
print(f"Valor base SHAP: {base_shap:.4f} | error de aditividad: {error_shap:.1e}")
importancias = importancia_permutacion.set_index("variable").join(
    shap_validacion.abs().mean().rename("SHAP_medio_absoluto"))
display(importancias.sort_values("SHAP_medio_absoluto", ascending=False).round(4))

fig, ax = plt.subplots(figsize=(7, 3.2))
s = importancias["SHAP_medio_absoluto"].sort_values()
ax.barh(s.index, s.values, color="#1F4E79")
ax.set_title("Importancia SHAP (validación): cuánto mueve cada dato la probabilidad")
plt.tight_layout(); plt.show()

## Parte 10 · El producto: lista de septiembre 2026 (igual que la app)
- **Modelo operativo:** el mismo modelo reentrenado con **todas** las membresías con resultado (1.564), con sus propias medianas. Es el que usa la app.
- **Cartera:** membresías que vencen en los 30 días siguientes al corte, sin pases diarios ni duraciones imposibles. Se excluye a quien ya renovó.
- **Nivel:** ALTO ≥ 0,70, MEDIO ≥ 0,50, BAJO < 0,50. **Motivos:** hasta 3 variables que suben el riesgo más de 0,005 según SHAP (nunca el mes de vencimiento).

In [ ]:
# 1) Modelo operativo
mediana_por_plan_op, mediana_general_op = calcular_medianas(analitico)
analitico_operativo = completar_monto_y_duracion(analitico, mediana_por_plan_op, mediana_general_op)
modelo_operativo = clone(modelo_seleccionado).fit(analitico_operativo[VARIABLES_MODELO], analitico_operativo["no_renovo"])

# 2) Cartera a vencer en el horizonte
FECHA_GENERACION, HORIZONTE_DIAS = fecha_corte, 30
cartera = datos.loc[(datos["fecha_fin"] > FECHA_GENERACION)
                    & (datos["fecha_fin"] <= FECHA_GENERACION + pd.Timedelta(days=HORIZONTE_DIAS))
                    & datos["plan"].ne("DIARIO")
                    & ~((datos["duracion_dias"] <= DURACION_MIN) | (datos["duracion_dias"] > DURACION_MAX))].copy()

def ya_renovo(fila):                           # ya registró una membresía posterior: no hace falta llamarlo
    h = actividades_por_socio.get(fila["socio_id"])
    return h is not None and bool(((h.index != fila.name) & (h["fecha_actividad"] > fila["fecha_actividad"])).any())

cartera["ya_renovo"] = cartera.apply(ya_renovo, axis=1)
cartera = cartera.join(calcular_historial(cartera, modelables))
cartera = completar_monto_y_duracion(cartera, mediana_por_plan_op, mediana_general_op)
a_priorizar = cartera.loc[~cartera["ya_renovo"]].copy()

# 3) Score, nivel y motivos
NOMBRE_VARIABLE = {"ciclos_previos": "Membresías previas", "antiguedad_dias": "Antigüedad (días)",
                   "duracion_modelo": "Duración del plan (días)", "monto_actual": "Monto actual (USD)",
                   "monto_historico": "Monto histórico (USD)",
                   "dias_desde_ultima_membresia": "Días desde la membresía anterior", "plan": "Plan"}

def nivel_riesgo(p):
    return "ALTO" if p >= UMBRAL_ALTO else ("MEDIO" if p >= UMBRAL_MEDIO else "BAJO")

def es(x, d=2, signo=False):                   # número con coma decimal: 0,05
    t = f"{x:{'+' if signo else ''},.{d}f}"
    return t.replace(",", "X").replace(".", ",").replace("X", ".")

def motivos(modelo, X):
    _, aportes, _ = explicar_shap(modelo, X)
    textos = []
    for idx, fila in aportes.iterrows():
        elegidas = sorted([v for v in NOMBRE_VARIABLE if fila[v] > 0.005], key=lambda v: fila[v], reverse=True)[:3]
        partes = []
        for v in elegidas:
            valor = X.at[idx, v]
            valor = "sin dato" if pd.isna(valor) else (f"{valor:,.0f}".replace(",", ".") if v in VARIABLES_NUMERICAS else str(valor))
            partes.append(f"{NOMBRE_VARIABLE[v]}: {valor} ({es(fila[v], 2, True)})")
        textos.append("; ".join(partes) if partes else "Sin factores de riesgo destacados")
    return pd.Series(textos, index=X.index)

a_priorizar["probabilidad"] = modelo_operativo.predict_proba(a_priorizar[VARIABLES_MODELO])[:, 1]
a_priorizar["nivel_riesgo"] = a_priorizar["probabilidad"].apply(nivel_riesgo)
a_priorizar["motivos"] = motivos(modelo_operativo, a_priorizar[VARIABLES_MODELO])
a_priorizar["codigo"] = a_priorizar["socio_id"].apply(codigo_socio)

lista = (a_priorizar.sort_values("probabilidad", ascending=False)
         .drop_duplicates("socio_id", keep="first").copy())     # una fila por socio (la de mayor riesgo)
lista["prioridad"] = np.arange(1, len(lista) + 1)
lista["dentro_de_capacidad"] = np.where(lista["prioridad"] <= TOP_K, "Sí", "No")
print(f"Lista generada con datos al {FECHA_GENERACION.date()}: {len(lista)} socios ->",
      lista["nivel_riesgo"].value_counts().to_dict())
display(lista[["prioridad", "codigo", "fecha_fin", "plan", "probabilidad", "nivel_riesgo", "motivos",
               "dentro_de_capacidad"]].head(TOP_K).round(3))

## Parte 11 · Controles: ¿este notebook da lo mismo que el original y que la app?
Si algún control falla, la simplificación cambió algo y hay que revisarla antes de usar el notebook.

In [ ]:
def control(descripcion, obtenido, esperado):
    ok = obtenido == esperado
    print(f"{'OK  ' if ok else 'FALLA'} {descripcion}: obtenido {obtenido} | esperado {esperado}")
    return ok

pr = comparacion_prueba.loc[modelo_seleccionado_nombre]
resultados_control = [
    control("Registros con identidad", len(datos), 2748),
    control("Socios distintos", datos["socio_id"].nunique(), 1410),
    control("Membresías evaluables", len(analitico), 1564),
    control("No renovaron", int(analitico["no_renovo"].sum()), 1098),
    control("Entrenamiento / validación / prueba", (len(train), len(validation), len(test)), (845, 247, 277)),
    control("Prueba anti-fuga con corte T−30", fuga_con_corte, 0),
    control("Modelo seleccionado", modelo_seleccionado_nombre, "Random Forest"),
    control("Umbral", umbral_seleccionado, 0.32),
    control("PR-AUC en prueba", round(pr["PR-AUC"], 4), 0.7934),
    control("Precisión@25 en prueba", round(pr[f"Precisión@{TOP_K}"], 2), 0.80),
    control("Matriz de confusión (VN, FP, FN, VP)", tuple(int(v) for v in matriz.to_numpy().ravel()), (18, 74, 5, 180)),
    control("Socios en la lista de septiembre", len(lista), 57),
    control("Niveles ALTO / MEDIO / BAJO", tuple(int((lista["nivel_riesgo"] == n).sum()) for n in ("ALTO", "MEDIO", "BAJO")), (2, 39, 16)),
]

# Comparación fila a fila con la lista que exportó el notebook original (si el archivo existe)
ruta_original = Path("..") / "salidas" / "mvp" / "lista_operativa_2026-09_top25.csv"
if ruta_original.is_file():
    original = pd.read_csv(ruta_original)
    propia = lista.head(len(original))
    iguales = (np.allclose(original["probabilidad"].to_numpy(), propia["probabilidad"].round(3).to_numpy(), atol=1e-9)
               and list(original["nivel_riesgo"]) == list(propia["nivel_riesgo"])
               and list(original["plan"]) == list(propia["plan"])
               and list(original["factores principales"]) == list(propia["motivos"]))
    resultados_control.append(control("Top 25 igual al notebook original (probabilidad, nivel, plan y motivos)", iguales, True))
print("\nTODOS LOS CONTROLES OK" if all(resultados_control) else "\nHAY CONTROLES QUE FALLAN: revisar antes de usar")

## Objetos disponibles para trabajar
Después de ejecutar todo, quedan en memoria y listos para las pruebas:

| Objeto | Qué es |
|---|---|
| `datos` | Todas las filas limpias, con estado y etiqueta |
| `modelables`, `analitico` | Membresías con variables en T−30 (todas / solo con resultado) |
| `train`, `validation`, `test` y sus `X_` / `y_` | Los tres conjuntos por fecha |
| `modelo_seleccionado`, `umbral_seleccionado` | Random Forest evaluado (entrenado solo con 2025) |
| `modelo_operativo` | El mismo modelo con todos los datos: **el que usa la app** |
| `mediana_por_plan`, `mediana_general` | Medianas de monto de entrenamiento |
| `explicar_shap(modelo, X)`, `motivos(modelo, X)`, `nivel_riesgo(p)` | Explicación, motivos y nivel |
| `lista` | Lista de septiembre con prioridad, nivel y motivos |
| `codigo_socio(socio_id)` | Código anónimo para mostrar socios sin datos personales |